# Chimera baselines — train cross-method integration latents

This notebook drives the **baseline matrix** for cross-method comparison
against Chimera. Each cell trains (or loads cached) a latent embedding
for one method on one dataset, saving `z_mean.npy` + `config.json` to
`chimera/training/runs/baselines_<method>/...`. Downstream metric
notebooks consume those files; this notebook does **not** compute
metrics or render UMAPs.

**Code surface** — implementation is split across:
- [`chimera/training/baseline_runner.py`](../training/baseline_runner.py): single-dataset trainer (CLI + Python API).
- [`chimera/training/run_all_baselines.py`](../training/run_all_baselines.py): orchestrator (importable + CLI).
- Spec: `chimera-private/baselines_prompt.md`.

**Operational rules** (enforced by the runner):
- **Rule A** — CPU only, 5 torch threads. No MPS, no CUDA.
- **Rule B** — `max_epochs ≤ 150`, default 100 (fairness cap vs Chimera).
- **Rule C** — sequential subprocesses (no parallel trainings).
- **Fair-playing-field** — no HVG selection, no `sc.pp.scale`, no per-method
  hyperparameter tuning. Each method's library defaults; only `n_latent=10`
  is forced equal where the API supports it.

**Tier-1 methods** (run for every dataset, in this order):
`pca → harmony → scvi → scanvi → sysvi`.

**Tier-2** (`camex`, `saturn`) are NotImplementedError stubs — they raise a
clean error rather than producing a half-working baseline.

The orchestrator's `skip-if-cached` logic makes the second top-to-bottom
run of this notebook complete in seconds. Re-execute freely; trainings
that have already completed are not redone.

## 2 — Environment setup

Kernel must be `Python (chimera_env)`. The cell below verifies torch threads, accelerator, and the active conda env. **STOP if any check fails** — the matrix will burn the wrong env otherwise.

In [1]:
import os
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

# `chimera/training/` is a script directory, not a Python package on
# this layout, so the orchestrator is imported via sys.path injection.
# The notebook lives at `chimera/notebooks/02_baselines.ipynb`, so
# `Path.cwd()` should be `chimera/notebooks/` and the training dir
# is the sibling.
NOTEBOOK_DIR = Path.cwd()
TRAINING_DIR = (NOTEBOOK_DIR.parent / "training").resolve()
if str(TRAINING_DIR) not in sys.path:
    sys.path.insert(0, str(TRAINING_DIR))

import run_all_baselines as orch  # type: ignore
from run_all_baselines import (  # type: ignore
    DATASETS, TIER1_METHODS, TIER2_METHODS,
    run_one, run_phase, run_all, write_summary, load_summary,
    is_cached, expected_cache_path,
)

print(f"orchestrator module: {orch.__file__}")
print(f"DATASETS: {len(DATASETS)} entries")
print(f"TIER1_METHODS: {TIER1_METHODS}")
print(f"TIER2_METHODS: {TIER2_METHODS}")
print(f"runs root: {orch.RUNS_ROOT}")
print(f"log root:  {orch.LOG_ROOT}")


orchestrator module: training/run_all_baselines.py
DATASETS: 7 entries
TIER1_METHODS: ['pca', 'harmony', 'scvi', 'scanvi', 'sysvi']
TIER2_METHODS: ['camex', 'saturn']
runs root: training/runs
log root:  training/runs/baselines_logs


In [2]:
# Hard checks — STOP if any fail.
# Cell 3 also enforces Rule A in the notebook process itself, matching
# the runner's module-init pattern. The runner subprocesses set the
# same caps independently; setting them here keeps any incidental
# torch ops in the notebook from out-running the cap.
import os
os.environ.setdefault("OMP_NUM_THREADS", "5")
os.environ.setdefault("MKL_NUM_THREADS", "5")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "5")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "5")

import torch  # noqa: E402
torch.set_num_threads(5)
try:
    torch.set_num_interop_threads(2)
except RuntimeError:
    # set_num_interop_threads must be called before any parallel work
    # has happened. If torch was already initialized by an earlier cell,
    # we can't change interop threads — that's fine, the runner
    # subprocess sets the right value at its own init.
    pass

print(f"torch.get_num_threads()       = {torch.get_num_threads()}  (target 5)")
print(f"torch.get_num_interop_threads = {torch.get_num_interop_threads()}  (target 2)")
print(f"torch.cuda.is_available()     = {torch.cuda.is_available()}  (must be False)")
mps_avail = getattr(getattr(torch.backends, 'mps', None), 'is_available', lambda: False)()
print(f"torch.backends.mps.is_available() = {mps_avail}  (we still use CPU)")
print(f"CONDA_DEFAULT_ENV = {os.environ.get('CONDA_DEFAULT_ENV')}  (must be 'chimera_env')")
print()
import importlib.metadata as md_meta
for pkg in ("scvi-tools", "anndata", "scanpy", "torch", "lightning", "harmonypy"):
    try:
        print(f"{pkg:14}: {md_meta.version(pkg)}")
    except md_meta.PackageNotFoundError:
        print(f"{pkg:14}: NOT INSTALLED")

env = os.environ.get('CONDA_DEFAULT_ENV')
if env != 'chimera_env':
    print()
    print('=' * 60)
    print('!! STOP — re-run from the chimera_env kernel.')
    print('   Current env:', env)
    print('=' * 60)


torch.get_num_threads()       = 5  (target 5)
torch.get_num_interop_threads = 2  (target 2)
torch.cuda.is_available()     = False  (must be False)
torch.backends.mps.is_available() = True  (we still use CPU)
CONDA_DEFAULT_ENV = chimera_env  (must be 'chimera_env')

scvi-tools    : 1.4.2
anndata       : 0.12.11
scanpy        : 1.12.1
torch         : 2.11.0
lightning     : 2.6.1
harmonypy     : 2.0.0


## 3 — Dataset matrix

Single source of truth: `DATASETS` in `run_all_baselines.py`. Editing
that list is the only way to change what runs.

**Routing rules**:
- **Pancreas** (`camex_*` except testis): `batch_col=batch`. In all
  pancreas datasets except H+Mo, batch is 1-to-1 with species; the
  runner detects this and adjusts SysVI's `categorical_covariate_keys`
  to avoid double-counting.
- **Testis** (`camex_testis_*`): `batch_col=sample_batch` — donor-level
  variation that's genuinely within-species.
- **Skipped**: `van_zyl_unified.h5ad` (refinement.md flags as not ready).

The cell below opens each h5ad in backed mode for a fast peek of
n_obs, n_vars, and the unique counts of the relevant `.obs` columns.

In [3]:
import anndata as ad

rows = []
for spec in DATASETS:
    p = spec.path_resolved
    a = ad.read_h5ad(p, backed='r')
    obs = a.obs
    rows.append({
        'stem': spec.stem,
        'n_obs': a.n_obs,
        'n_vars': a.n_vars,
        'species_col': spec.species_col,
        'species_levels': sorted(obs[spec.species_col].unique().tolist())
            if spec.species_col in obs.columns else None,
        'batch_col': spec.batch_col,
        'n_batches': obs[spec.batch_col].nunique() if spec.batch_col in obs.columns else None,
        'labels_col': spec.labels_col,
        'n_celltypes': obs[spec.labels_col].nunique() if spec.labels_col in obs.columns else None,
        'has_sample_batch': 'sample_batch' in obs.columns,
        'note': spec.note,
    })
df = pd.DataFrame(rows)
df


,stem,n_obs,n_vars,species_col,species_levels,batch_col,n_batches,labels_col,n_celltypes,has_sample_batch,note
0,camex_human_monkey,12608,8430,species,"[Human, Monkey]",batch,3,cell_type,8,False,2-sp pancreas; H+Mo. batch carries 1 within-sp...
1,camex_human_mouse,8091,8257,species,"[Human, Mouse]",batch,2,cell_type,8,False,2-sp pancreas; H+Mouse. batch ≡ species 1-to-1.
2,camex_human_zebrafish,9655,7998,species,"[Human, Zebrafish]",batch,2,cell_type,5,False,2-sp pancreas; H+Zebrafish. batch ≡ species 1-...
3,camex_pancreas_unified,22104,8879,species,"[Human, Monkey, Mouse, Zebrafish]",batch,5,cell_type,9,False,4-sp pancreas. batch ≡ species 1-to-1.
4,camex_testis_human_macaque,23593,14640,species,"[Human, Macaque]",sample_batch,7,cell_type,10,True,2-sp testis; H+Mac. sample_batch is donor-leve...
5,camex_testis_human_mouse,21241,14731,species,"[Human, Mouse]",sample_batch,7,cell_type,13,True,2-sp testis; H+Mouse. sample_batch is donor-le...
6,camex_testis_unified,28854,15732,species,"[Human, Macaque, Mouse]",sample_batch,9,cell_type,13,True,3-sp testis; H+Mac+Mouse. sample_batch is dono...


## 4 — Cache state

Where every (dataset, method, seed=42) baseline currently is. ✓ = cached
on disk; ✗ = will be trained when the matching cell runs. **No training
in this cell** — purely a status read.

If a row is mostly ✗, expect substantial CPU wall-time when the per-method
sections below run (per the runner's wall-time table: scVI / scANVI /
SysVI on the larger datasets are ~30-90 minutes each on CPU).

In [4]:
status = {}
for spec in DATASETS:
    row = {}
    for m in TIER1_METHODS:
        row[m] = '✓' if is_cached(spec, m, seed=42) else '✗'
    status[spec.stem] = row
df_status = pd.DataFrame(status).T
df_status


,pca,harmony,scvi,scanvi,sysvi
camex_human_monkey,✓,✓,✓,✓,✓
camex_human_mouse,✓,✓,✓,✓,✓
camex_human_zebrafish,✓,✓,✓,✓,✓
camex_pancreas_unified,✓,✓,✓,✓,✓
camex_testis_human_macaque,✓,✓,✓,✓,✓
camex_testis_human_mouse,✓,✓,✓,✓,✓
camex_testis_unified,✓,✓,✓,✓,✓


## 5a — Train: PCA (no integration)

Log-norm + PCA on the full gene set, no HVG, no scaling. The no-integration reference for downstream metrics. Deterministic given the data, so the cache directory omits the seed.

**Runtime per dataset**: seconds.
**Look for**: `latent_shape == (n_obs, 10)`, no NaN.

In [5]:
# pca on camex_human_monkey
spec_obj = next(s for s in DATASETS if s.stem == 'camex_human_monkey')
r = run_one(spec_obj, 'pca', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,049 [INFO] SKIP cached: camex_human_monkey / pca / s42 → training/runs/baselines_pca/camex_human_monkey/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_human_monkey,pca,42,SKIPPED,0.0,"(12608, 10)",./...,None,None


In [6]:
# pca on camex_human_mouse
spec_obj = next(s for s in DATASETS if s.stem == 'camex_human_mouse')
r = run_one(spec_obj, 'pca', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,073 [INFO] SKIP cached: camex_human_mouse / pca / s42 → training/runs/baselines_pca/camex_human_mouse/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_human_mouse,pca,42,SKIPPED,0.0,"(8091, 10)",./...,None,None


In [7]:
# pca on camex_human_zebrafish
spec_obj = next(s for s in DATASETS if s.stem == 'camex_human_zebrafish')
r = run_one(spec_obj, 'pca', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,087 [INFO] SKIP cached: camex_human_zebrafish / pca / s42 → training/runs/baselines_pca/camex_human_zebrafish/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_human_zebrafish,pca,42,SKIPPED,0.0,"(9655, 10)",./...,None,None


In [8]:
# pca on camex_pancreas_unified
spec_obj = next(s for s in DATASETS if s.stem == 'camex_pancreas_unified')
r = run_one(spec_obj, 'pca', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,103 [INFO] SKIP cached: camex_pancreas_unified / pca / s42 → training/runs/baselines_pca/camex_pancreas_unified/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_pancreas_unified,pca,42,SKIPPED,0.0,"(22104, 10)",./...,None,None


In [9]:
# pca on camex_testis_human_macaque
spec_obj = next(s for s in DATASETS if s.stem == 'camex_testis_human_macaque')
r = run_one(spec_obj, 'pca', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,121 [INFO] SKIP cached: camex_testis_human_macaque / pca / s42 → training/runs/baselines_pca/camex_testis_human_macaque/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_testis_human_macaque,pca,42,SKIPPED,0.0,"(23593, 10)",./...,None,None


In [10]:
# pca on camex_testis_human_mouse
spec_obj = next(s for s in DATASETS if s.stem == 'camex_testis_human_mouse')
r = run_one(spec_obj, 'pca', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,139 [INFO] SKIP cached: camex_testis_human_mouse / pca / s42 → training/runs/baselines_pca/camex_testis_human_mouse/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_testis_human_mouse,pca,42,SKIPPED,0.0,"(21241, 10)",./...,None,None


In [11]:
# pca on camex_testis_unified
spec_obj = next(s for s in DATASETS if s.stem == 'camex_testis_unified')
r = run_one(spec_obj, 'pca', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,154 [INFO] SKIP cached: camex_testis_unified / pca / s42 → training/runs/baselines_pca/camex_testis_unified/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_testis_unified,pca,42,SKIPPED,0.0,"(28854, 10)",./...,None,None


In [12]:
# Summary across datasets for method='pca'
results_pca = []
for spec in DATASETS:
    cp = expected_cache_path(spec, 'pca', seed=42)
    cached = cp.exists()
    cfg_path = cp.parent / 'config.json' if cached else None
    cfg = json.loads(cfg_path.read_text()) if cfg_path and cfg_path.exists() else {}
    results_pca.append({
        'dataset': spec.stem,
        'cached':  cached,
        'shape':   tuple(cfg.get('latent_shape', [])) or None,
        'n_obs':   cfg.get('n_obs'),
        'batch_key': cfg.get('batch_key'),
        'cache_path': str(cp) if cached else None,
    })
pd.DataFrame(results_pca)


,dataset,cached,shape,n_obs,batch_key,cache_path
0,camex_human_monkey,True,"(12608, 10)",12608,None,./...
1,camex_human_mouse,True,"(8091, 10)",8091,None,./...
2,camex_human_zebrafish,True,"(9655, 10)",9655,None,./...
3,camex_pancreas_unified,True,"(22104, 10)",22104,None,./...
4,camex_testis_human_macaque,True,"(23593, 10)",23593,None,./...
5,camex_testis_human_mouse,True,"(21241, 10)",21241,None,./...
6,camex_testis_unified,True,"(28854, 10)",28854,None,./...


## 5b — Train: Harmony

Log-norm + 10-d PCA + harmonypy.run_harmony. All theta/lamb/sigma/nclust at library defaults (fair-playing-field rule). harmonypy v2.0.0 returns `Z_corr` already in `(n_cells, n_components)` shape; the runner shape-checks defensively against the older `(n_components, n_cells)` convention.

**Runtime per dataset**: seconds to a couple of minutes.
**Look for**: latent != raw PCA (Harmony actually moved the embedding).

In [13]:
# harmony on camex_human_monkey
spec_obj = next(s for s in DATASETS if s.stem == 'camex_human_monkey')
r = run_one(spec_obj, 'harmony', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,179 [INFO] SKIP cached: camex_human_monkey / harmony / s42 → training/runs/baselines_harmony/camex_human_monkey-s42-bk_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_human_monkey,harmony,42,SKIPPED,0.0,"(12608, 10)",./...,None,None


In [14]:
# harmony on camex_human_mouse
spec_obj = next(s for s in DATASETS if s.stem == 'camex_human_mouse')
r = run_one(spec_obj, 'harmony', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,196 [INFO] SKIP cached: camex_human_mouse / harmony / s42 → training/runs/baselines_harmony/camex_human_mouse-s42-bk_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_human_mouse,harmony,42,SKIPPED,0.0,"(8091, 10)",./...,None,None


In [15]:
# harmony on camex_human_zebrafish
spec_obj = next(s for s in DATASETS if s.stem == 'camex_human_zebrafish')
r = run_one(spec_obj, 'harmony', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,209 [INFO] SKIP cached: camex_human_zebrafish / harmony / s42 → training/runs/baselines_harmony/camex_human_zebrafish-s42-bk_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_human_zebrafish,harmony,42,SKIPPED,0.0,"(9655, 10)",./...,None,None


In [16]:
# harmony on camex_pancreas_unified
spec_obj = next(s for s in DATASETS if s.stem == 'camex_pancreas_unified')
r = run_one(spec_obj, 'harmony', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,224 [INFO] SKIP cached: camex_pancreas_unified / harmony / s42 → training/runs/baselines_harmony/camex_pancreas_unified-s42-bk_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_pancreas_unified,harmony,42,SKIPPED,0.0,"(22104, 10)",./...,None,None


In [17]:
# harmony on camex_testis_human_macaque
spec_obj = next(s for s in DATASETS if s.stem == 'camex_testis_human_macaque')
r = run_one(spec_obj, 'harmony', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,250 [INFO] SKIP cached: camex_testis_human_macaque / harmony / s42 → training/runs/baselines_harmony/camex_testis_human_macaque-s42-bk_sample_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_testis_human_macaque,harmony,42,SKIPPED,0.0,"(23593, 10)",./...,None,None


In [18]:
# harmony on camex_testis_human_mouse
spec_obj = next(s for s in DATASETS if s.stem == 'camex_testis_human_mouse')
r = run_one(spec_obj, 'harmony', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,265 [INFO] SKIP cached: camex_testis_human_mouse / harmony / s42 → training/runs/baselines_harmony/camex_testis_human_mouse-s42-bk_sample_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_testis_human_mouse,harmony,42,SKIPPED,0.0,"(21241, 10)",./...,None,None


In [19]:
# harmony on camex_testis_unified
spec_obj = next(s for s in DATASETS if s.stem == 'camex_testis_unified')
r = run_one(spec_obj, 'harmony', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,280 [INFO] SKIP cached: camex_testis_unified / harmony / s42 → training/runs/baselines_harmony/camex_testis_unified-s42-bk_sample_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_testis_unified,harmony,42,SKIPPED,0.0,"(28854, 10)",./...,None,None


In [20]:
# Summary across datasets for method='harmony'
results_harmony = []
for spec in DATASETS:
    cp = expected_cache_path(spec, 'harmony', seed=42)
    cached = cp.exists()
    cfg_path = cp.parent / 'config.json' if cached else None
    cfg = json.loads(cfg_path.read_text()) if cfg_path and cfg_path.exists() else {}
    results_harmony.append({
        'dataset': spec.stem,
        'cached':  cached,
        'shape':   tuple(cfg.get('latent_shape', [])) or None,
        'n_obs':   cfg.get('n_obs'),
        'batch_key': cfg.get('batch_key'),
        'cache_path': str(cp) if cached else None,
    })
pd.DataFrame(results_harmony)


,dataset,cached,shape,n_obs,batch_key,cache_path
0,camex_human_monkey,True,"(12608, 10)",12608,batch,./...
1,camex_human_mouse,True,"(8091, 10)",8091,batch,./...
2,camex_human_zebrafish,True,"(9655, 10)",9655,batch,./...
3,camex_pancreas_unified,True,"(22104, 10)",22104,batch,./...
4,camex_testis_human_macaque,True,"(23593, 10)",23593,sample_batch,./...
5,camex_testis_human_mouse,True,"(21241, 10)",21241,sample_batch,./...
6,camex_testis_unified,True,"(28854, 10)",28854,sample_batch,./...


## 5c — Train: scVI

Vanilla scVI with raw counts in `.X` and `batch_key=batch` (pancreas) or `batch_key=sample_batch` (testis). Reuses the `compare_vs_scvi.py::get_or_train_scvi` helper so the existing `runs/baselines_scvi/` cache is used directly when present. `max_epochs=100, n_latent=10, n_hidden=128` per the prompt's exception note (Chimera's E8 gates are calibrated against these).

**Runtime per dataset**: 30-90 min CPU on a fresh train, ~0s on cache hit.
**Look for**: training loss decreases monotonically in the log; if the matrix already had cached scVI latents, this section completes in seconds (every cell shows SKIPPED).

In [21]:
# scvi on camex_human_monkey
spec_obj = next(s for s in DATASETS if s.stem == 'camex_human_monkey')
r = run_one(spec_obj, 'scvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,298 [INFO] SKIP cached: camex_human_monkey / scvi / s42 → training/runs/baselines_scvi/camex_human_monkey-s42-e100-nl10-bk_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_human_monkey,scvi,42,SKIPPED,0.0,"(12608, 10)",./...,None,None


In [22]:
# scvi on camex_human_mouse
spec_obj = next(s for s in DATASETS if s.stem == 'camex_human_mouse')
r = run_one(spec_obj, 'scvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,307 [INFO] SKIP cached: camex_human_mouse / scvi / s42 → training/runs/baselines_scvi/camex_human_mouse-s42-e100-nl10-bk_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_human_mouse,scvi,42,SKIPPED,0.0,"(8091, 10)",./...,None,None


In [23]:
# scvi on camex_human_zebrafish
spec_obj = next(s for s in DATASETS if s.stem == 'camex_human_zebrafish')
r = run_one(spec_obj, 'scvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,317 [INFO] SKIP cached: camex_human_zebrafish / scvi / s42 → training/runs/baselines_scvi/camex_human_zebrafish-s42-e100-nl10-bk_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_human_zebrafish,scvi,42,SKIPPED,0.0,"(9655, 10)",./...,None,None


In [24]:
# scvi on camex_pancreas_unified
spec_obj = next(s for s in DATASETS if s.stem == 'camex_pancreas_unified')
r = run_one(spec_obj, 'scvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,326 [INFO] SKIP cached: camex_pancreas_unified / scvi / s42 → training/runs/baselines_scvi/camex_pancreas_unified-s42-e100-nl10-bk_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_pancreas_unified,scvi,42,SKIPPED,0.0,"(22104, 10)",./...,None,None


In [25]:
# scvi on camex_testis_human_macaque
spec_obj = next(s for s in DATASETS if s.stem == 'camex_testis_human_macaque')
r = run_one(spec_obj, 'scvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,336 [INFO] SKIP cached: camex_testis_human_macaque / scvi / s42 → training/runs/baselines_scvi/camex_testis_human_macaque-s42-e100-nl10-bk_sample_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_testis_human_macaque,scvi,42,SKIPPED,0.0,"(23593, 10)",./...,None,None


In [26]:
# scvi on camex_testis_human_mouse
spec_obj = next(s for s in DATASETS if s.stem == 'camex_testis_human_mouse')
r = run_one(spec_obj, 'scvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,345 [INFO] SKIP cached: camex_testis_human_mouse / scvi / s42 → training/runs/baselines_scvi/camex_testis_human_mouse-s42-e100-nl10-bk_sample_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_testis_human_mouse,scvi,42,SKIPPED,0.0,"(21241, 10)",./...,None,None


In [27]:
# scvi on camex_testis_unified
spec_obj = next(s for s in DATASETS if s.stem == 'camex_testis_unified')
r = run_one(spec_obj, 'scvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,355 [INFO] SKIP cached: camex_testis_unified / scvi / s42 → training/runs/baselines_scvi/camex_testis_unified-s42-e100-nl10-bk_sample_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_testis_unified,scvi,42,SKIPPED,0.0,"(28854, 10)",./...,None,None


In [28]:
# Summary across datasets for method='scvi'
results_scvi = []
for spec in DATASETS:
    cp = expected_cache_path(spec, 'scvi', seed=42)
    cached = cp.exists()
    cfg_path = cp.parent / 'config.json' if cached else None
    cfg = json.loads(cfg_path.read_text()) if cfg_path and cfg_path.exists() else {}
    results_scvi.append({
        'dataset': spec.stem,
        'cached':  cached,
        'shape':   tuple(cfg.get('latent_shape', [])) or None,
        'n_obs':   cfg.get('n_obs'),
        'batch_key': cfg.get('batch_key'),
        'cache_path': str(cp) if cached else None,
    })
pd.DataFrame(results_scvi)


,dataset,cached,shape,n_obs,batch_key,cache_path
0,camex_human_monkey,True,None,None,batch,./...
1,camex_human_mouse,True,None,None,batch,./...
2,camex_human_zebrafish,True,None,None,batch,./...
3,camex_pancreas_unified,True,None,None,batch,./...
4,camex_testis_human_macaque,True,None,None,sample_batch,./...
5,camex_testis_human_mouse,True,None,None,sample_batch,./...
6,camex_testis_unified,True,None,None,sample_batch,./...


## 5d — Train: scANVI

Two-step recipe: SCVI pretrain (100 epochs) → `SCANVI.from_scvi_model(unlabeled_category='Unknown')` → `SCANVI.train(max_epochs=20, n_samples_per_label=100)`. Sum 120, well under the 150 cap.

Reference: scvi-tools harmonization tutorial. The tutorial uses `unlabeled_category='Unknown'` (the prompt's `'UNK'` is overridden in favor of canonical per the prompt's own "trust the canonical reference" rule).

**Runtime per dataset**: 30-90 min CPU.
**Look for**: scANVI loss decreases monotonically; if it spikes, the labels_key is mis-wired. scANVI typically *under*-mixes species relative to scVI when given labels — high celltype silhouette, moderate species mixing. The downstream metrics step will check this.

In [29]:
# scanvi on camex_human_monkey
spec_obj = next(s for s in DATASETS if s.stem == 'camex_human_monkey')
r = run_one(spec_obj, 'scanvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,372 [INFO] SKIP cached: camex_human_monkey / scanvi / s42 → training/runs/baselines_scanvi/camex_human_monkey-s42-bk_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_human_monkey,scanvi,42,SKIPPED,0.0,"(12608, 10)",./...,None,None


In [30]:
# scanvi on camex_human_mouse
spec_obj = next(s for s in DATASETS if s.stem == 'camex_human_mouse')
r = run_one(spec_obj, 'scanvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,382 [INFO] SKIP cached: camex_human_mouse / scanvi / s42 → training/runs/baselines_scanvi/camex_human_mouse-s42-bk_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_human_mouse,scanvi,42,SKIPPED,0.0,"(8091, 10)",./...,None,None


In [31]:
# scanvi on camex_human_zebrafish
spec_obj = next(s for s in DATASETS if s.stem == 'camex_human_zebrafish')
r = run_one(spec_obj, 'scanvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,395 [INFO] SKIP cached: camex_human_zebrafish / scanvi / s42 → training/runs/baselines_scanvi/camex_human_zebrafish-s42-bk_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_human_zebrafish,scanvi,42,SKIPPED,0.0,"(9655, 10)",./...,None,None


In [32]:
# scanvi on camex_pancreas_unified
spec_obj = next(s for s in DATASETS if s.stem == 'camex_pancreas_unified')
r = run_one(spec_obj, 'scanvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,404 [INFO] SKIP cached: camex_pancreas_unified / scanvi / s42 → training/runs/baselines_scanvi/camex_pancreas_unified-s42-bk_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_pancreas_unified,scanvi,42,SKIPPED,0.0,"(22104, 10)",./...,None,None


In [33]:
# scanvi on camex_testis_human_macaque
spec_obj = next(s for s in DATASETS if s.stem == 'camex_testis_human_macaque')
r = run_one(spec_obj, 'scanvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,413 [INFO] SKIP cached: camex_testis_human_macaque / scanvi / s42 → training/runs/baselines_scanvi/camex_testis_human_macaque-s42-bk_sample_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_testis_human_macaque,scanvi,42,SKIPPED,0.0,"(23593, 10)",./...,None,None


In [34]:
# scanvi on camex_testis_human_mouse
spec_obj = next(s for s in DATASETS if s.stem == 'camex_testis_human_mouse')
r = run_one(spec_obj, 'scanvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,422 [INFO] SKIP cached: camex_testis_human_mouse / scanvi / s42 → training/runs/baselines_scanvi/camex_testis_human_mouse-s42-bk_sample_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_testis_human_mouse,scanvi,42,SKIPPED,0.0,"(21241, 10)",./...,None,None


In [35]:
# scanvi on camex_testis_unified
spec_obj = next(s for s in DATASETS if s.stem == 'camex_testis_unified')
r = run_one(spec_obj, 'scanvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,432 [INFO] SKIP cached: camex_testis_unified / scanvi / s42 → training/runs/baselines_scanvi/camex_testis_unified-s42-bk_sample_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_testis_unified,scanvi,42,SKIPPED,0.0,"(28854, 10)",./...,None,None


In [36]:
# Summary across datasets for method='scanvi'
results_scanvi = []
for spec in DATASETS:
    cp = expected_cache_path(spec, 'scanvi', seed=42)
    cached = cp.exists()
    cfg_path = cp.parent / 'config.json' if cached else None
    cfg = json.loads(cfg_path.read_text()) if cfg_path and cfg_path.exists() else {}
    results_scanvi.append({
        'dataset': spec.stem,
        'cached':  cached,
        'shape':   tuple(cfg.get('latent_shape', [])) or None,
        'n_obs':   cfg.get('n_obs'),
        'batch_key': cfg.get('batch_key'),
        'cache_path': str(cp) if cached else None,
    })
pd.DataFrame(results_scanvi)


,dataset,cached,shape,n_obs,batch_key,cache_path
0,camex_human_monkey,True,"(12608, 10)",12608,batch,./...
1,camex_human_mouse,True,"(8091, 10)",8091,batch,./...
2,camex_human_zebrafish,True,"(9655, 10)",9655,batch,./...
3,camex_pancreas_unified,True,"(22104, 10)",22104,batch,./...
4,camex_testis_human_macaque,True,"(23593, 10)",23593,sample_batch,./...
5,camex_testis_human_mouse,True,"(21241, 10)",21241,sample_batch,./...
6,camex_testis_unified,True,"(28854, 10)",28854,sample_batch,./...


## 5e — Train: SysVI

Cross-system VampPrior cVAE with latent cycle-consistency. Now lives in scvi-tools as `scvi.external.SysVI` (the standalone `theislab/cross_system_integration` repo is unmaintained per the Theislab team).

**Routing**: `batch_key=species` (the system to mix); `categorical_covariate_keys=[batch]` (within-system batch) ONLY when batch carries genuine within-species variation (the runner detects this — pancreas's batch ≡ species 1-to-1 except H+Mo gets the covariate dropped automatically).

Preprocessing: `normalize_total + log1p` on a copy of the input (SysVI's docstring requires log-normalized expression in `.X`). `n_latent=10` forwarded through `**model_kwargs` (SysVI's library default is 15).

**Runtime per dataset**: 30-90 min CPU, potentially longer on the larger testis datasets.
**Look for**: SysVI is designed to mix species *while preserving* celltype — both species kNN-purity ↓ and silhouette-celltype maintained. The downstream metrics step will check this.

In [37]:
# sysvi on camex_human_monkey
spec_obj = next(s for s in DATASETS if s.stem == 'camex_human_monkey')
r = run_one(spec_obj, 'sysvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,453 [INFO] SKIP cached: camex_human_monkey / sysvi / s42 → training/runs/baselines_sysvi/camex_human_monkey-s42-bk_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_human_monkey,sysvi,42,SKIPPED,0.0,"(12608, 10)",./...,None,None


In [38]:
# sysvi on camex_human_mouse
spec_obj = next(s for s in DATASETS if s.stem == 'camex_human_mouse')
r = run_one(spec_obj, 'sysvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,462 [INFO] SKIP cached: camex_human_mouse / sysvi / s42 → training/runs/baselines_sysvi/camex_human_mouse-s42-bk_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_human_mouse,sysvi,42,SKIPPED,0.0,"(8091, 10)",./...,None,None


In [39]:
# sysvi on camex_human_zebrafish
spec_obj = next(s for s in DATASETS if s.stem == 'camex_human_zebrafish')
r = run_one(spec_obj, 'sysvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,473 [INFO] SKIP cached: camex_human_zebrafish / sysvi / s42 → training/runs/baselines_sysvi/camex_human_zebrafish-s42-bk_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_human_zebrafish,sysvi,42,SKIPPED,0.0,"(9655, 10)",./...,None,None


In [40]:
# sysvi on camex_pancreas_unified
spec_obj = next(s for s in DATASETS if s.stem == 'camex_pancreas_unified')
r = run_one(spec_obj, 'sysvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,483 [INFO] SKIP cached: camex_pancreas_unified / sysvi / s42 → training/runs/baselines_sysvi/camex_pancreas_unified-s42-bk_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_pancreas_unified,sysvi,42,SKIPPED,0.0,"(22104, 10)",./...,None,None


In [41]:
# sysvi on camex_testis_human_macaque
spec_obj = next(s for s in DATASETS if s.stem == 'camex_testis_human_macaque')
r = run_one(spec_obj, 'sysvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,493 [INFO] SKIP cached: camex_testis_human_macaque / sysvi / s42 → training/runs/baselines_sysvi/camex_testis_human_macaque-s42-bk_sample_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_testis_human_macaque,sysvi,42,SKIPPED,0.0,"(23593, 10)",./...,None,None


In [42]:
# sysvi on camex_testis_human_mouse
spec_obj = next(s for s in DATASETS if s.stem == 'camex_testis_human_mouse')
r = run_one(spec_obj, 'sysvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,507 [INFO] SKIP cached: camex_testis_human_mouse / sysvi / s42 → training/runs/baselines_sysvi/camex_testis_human_mouse-s42-bk_sample_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_testis_human_mouse,sysvi,42,SKIPPED,0.0,"(21241, 10)",./...,None,None


In [43]:
# sysvi on camex_testis_unified
spec_obj = next(s for s in DATASETS if s.stem == 'camex_testis_unified')
r = run_one(spec_obj, 'sysvi', seed=42)
pd.DataFrame([{
    'dataset':      r.dataset_stem,
    'method':       r.method,
    'seed':         r.seed,
    'status':       r.status,
    'wall_s':       round(r.wall_time_s, 1),
    'shape':        str(r.latent_shape),
    'cache_path':   r.cache_path,
    'log_path':     r.log_path,
    'error_msg':    r.error_msg,
}])


2026-04-27 00:44:53,516 [INFO] SKIP cached: camex_testis_unified / sysvi / s42 → training/runs/baselines_sysvi/camex_testis_unified-s42-bk_sample_batch/z_mean.npy


,dataset,method,seed,status,wall_s,shape,cache_path,log_path,error_msg
0,camex_testis_unified,sysvi,42,SKIPPED,0.0,"(28854, 10)",./...,None,None


In [44]:
# Summary across datasets for method='sysvi'
results_sysvi = []
for spec in DATASETS:
    cp = expected_cache_path(spec, 'sysvi', seed=42)
    cached = cp.exists()
    cfg_path = cp.parent / 'config.json' if cached else None
    cfg = json.loads(cfg_path.read_text()) if cfg_path and cfg_path.exists() else {}
    results_sysvi.append({
        'dataset': spec.stem,
        'cached':  cached,
        'shape':   tuple(cfg.get('latent_shape', [])) or None,
        'n_obs':   cfg.get('n_obs'),
        'batch_key': cfg.get('batch_key'),
        'cache_path': str(cp) if cached else None,
    })
pd.DataFrame(results_sysvi)


,dataset,cached,shape,n_obs,batch_key,cache_path
0,camex_human_monkey,True,"(12608, 10)",12608,batch,./...
1,camex_human_mouse,True,"(8091, 10)",8091,batch,./...
2,camex_human_zebrafish,True,"(9655, 10)",9655,batch,./...
3,camex_pancreas_unified,True,"(22104, 10)",22104,batch,./...
4,camex_testis_human_macaque,True,"(23593, 10)",23593,sample_batch,./...
5,camex_testis_human_mouse,True,"(21241, 10)",21241,sample_batch,./...
6,camex_testis_unified,True,"(28854, 10)",28854,sample_batch,./...


## 6 — Tier-2 (deferred): CAMEX, SATURN

These are clean `NotImplementedError` stubs in the runner — invoking
them produces a precise error message naming the gating step rather
than a half-working baseline. The cells below `try / except` so the
notebook still runs to completion; the error is rendered as cell
output, not as a traceback.

Status: **deferred** until Tier-1 is fully validated end-to-end. The
gating steps for each:
- **CAMEX**: clone the CAMEX codebase + document its preprocessing
  / training entry-point.
- **SATURN**: install snap-stanford/SATURN + precompute ESM-2 protein
  embeddings per species + reshape our concatenated cross-species
  adatas into the per-species form SATURN expects.

In [45]:
# camex (Tier-2 stub) — expected to raise NotImplementedError
for spec in DATASETS[:1]:  # one dataset is enough to confirm the stub shape
    r = run_one(spec, 'camex', seed=42)
    print(f'{spec.stem}: status={r.status}  err={r.error_msg}')


2026-04-27 00:44:53,532 [INFO] RUN  camex_human_monkey / camex / s42 → log=training/runs/baselines_logs/camex-camex_human_monkey-s42-bk_batch.log


2026-04-27 00:44:53,533 [INFO]     cmd: $MAMBA_ROOT_PREFIX/envs/chimera_env/bin/python3.12 training/baseline_runner.py --adata data/processed/camex_human_monkey.h5ad --method camex --seed 42 --species-key species --batch-key batch


2026-04-27 00:44:54,931 [ERROR] FAIL camex_human_monkey / camex / s42 (rc=1): camex: NotImplementedError: CAMEX baseline is not wired in this runner. Gating step: clone CAMEX from its source repository and document its preprocessing + training entry-point in `chimera-private/baselines_prompt.md` Tier-2 section. Until that is done, this stub deliberately raises so the matrix records `FAILED` for camex rather than silently skipping.


camex_human_monkey: status=NOT_IMPL  err=camex: NotImplementedError: CAMEX baseline is not wired in this runner. Gating step: clone CAMEX from its source repository and document its preprocessing + training entry-point in `chimera-private/baselines_prompt.md` Tier-2 section. Until that is done, this stub deliberately raises so the matrix records `FAILED` for camex rather than silently skipping.


In [46]:
# saturn (Tier-2 stub) — expected to raise NotImplementedError
for spec in DATASETS[:1]:  # one dataset is enough to confirm the stub shape
    r = run_one(spec, 'saturn', seed=42)
    print(f'{spec.stem}: status={r.status}  err={r.error_msg}')


2026-04-27 00:44:54,938 [INFO] RUN  camex_human_monkey / saturn / s42 → log=training/runs/baselines_logs/saturn-camex_human_monkey-s42-bk_batch.log


2026-04-27 00:44:54,938 [INFO]     cmd: $MAMBA_ROOT_PREFIX/envs/chimera_env/bin/python3.12 training/baseline_runner.py --adata data/processed/camex_human_monkey.h5ad --method saturn --seed 42 --species-key species --batch-key batch


2026-04-27 00:44:56,175 [ERROR] FAIL camex_human_monkey / saturn / s42 (rc=1): saturn: NotImplementedError: SATURN baseline is not wired in this runner. Gating step: install snap-stanford/SATURN and precompute ESM-2 protein embeddings per species. SATURN expects per-species adatas + species-specific protein embedding files, which is incompatible with our concatenated cross-species adata layout. Document the split + embedding pipeline in `chimera-private/baselines_prompt.md` Tier-2 before stubbing this out.


camex_human_monkey: status=NOT_IMPL  err=saturn: NotImplementedError: SATURN baseline is not wired in this runner. Gating step: install snap-stanford/SATURN and precompute ESM-2 protein embeddings per species. SATURN expects per-species adatas + species-specific protein embedding files, which is incompatible with our concatenated cross-species adata layout. Document the split + embedding pipeline in `chimera-private/baselines_prompt.md` Tier-2 before stubbing this out.


## 7 — Summary

The orchestrator persists a `_summary.json` after every CLI invocation
(`run_all_baselines.py`). When the notebook drives the matrix cell-by-
cell, we re-derive the summary from the per-method `config.json` files
on disk so the table reflects the current cache state regardless of
how the trainings were launched.

In [47]:
rows = []
for spec in DATASETS:
    for method in TIER1_METHODS + TIER2_METHODS:
        cp = expected_cache_path(spec, method, seed=42)
        cached = cp.exists()
        cfg_path = cp.parent / 'config.json' if cached else None
        cfg = json.loads(cfg_path.read_text()) if cfg_path and cfg_path.exists() else {}
        rows.append({
            'dataset': spec.stem,
            'method':  method,
            'tier':    1 if method in TIER1_METHODS else 2,
            'status':  'OK' if cached else ('—' if method in TIER2_METHODS else 'PENDING'),
            'shape':   tuple(cfg.get('latent_shape', [])) or None,
            'n_latent_actual': cfg.get('n_latent_actual'),
            'n_latent_override': cfg.get('n_latent_override'),
            'device':  cfg.get('device'),
            'max_epochs': cfg.get('max_epochs'),
            'cache_path': str(cp) if cached else None,
        })
df_final = pd.DataFrame(rows)

def _highlight(row):
    if row['status'] == 'OK':
        return ['background-color: #e8f5e9'] * len(row)  # green
    if row['status'] == 'PENDING':
        return ['background-color: #fff8e1'] * len(row)  # amber
    return ['color: #999'] * len(row)

df_final.style.apply(_highlight, axis=1)


,dataset,method,tier,status,shape,n_latent_actual,n_latent_override,device,max_epochs,cache_path
0,camex_human_monkey,pca,1,OK,"(12608, 10)",10.000000,True,cpu,nan,training/runs/baselines_pca/camex_human_monkey/z_mean.npy
1,camex_human_monkey,harmony,1,OK,"(12608, 10)",10.000000,True,cpu,nan,training/runs/baselines_harmony/camex_human_monkey-s42-bk_batch/z_mean.npy
2,camex_human_monkey,scvi,1,OK,None,nan,None,None,100.000000,training/runs/baselines_scvi/camex_human_monkey-s42-e100-nl10-bk_batch/z_mean.npy
3,camex_human_monkey,scanvi,1,OK,"(12608, 10)",10.000000,True,cpu,120.000000,training/runs/baselines_scanvi/camex_human_monkey-s42-bk_batch/z_mean.npy
4,camex_human_monkey,sysvi,1,OK,"(12608, 10)",10.000000,True,cpu,100.000000,training/runs/baselines_sysvi/camex_human_monkey-s42-bk_batch/z_mean.npy
5,camex_human_monkey,camex,2,—,None,nan,None,None,nan,None
6,camex_human_monkey,saturn,2,—,None,nan,None,None,nan,None
7,camex_human_mouse,pca,1,OK,"(8091, 10)",10.000000,True,cpu,nan,training/runs/baselines_pca/camex_human_mouse/z_mean.npy
8,camex_human_mouse,harmony,1,OK,"(8091, 10)",10.000000,True,cpu,nan,training/runs/baselines_harmony/camex_human_mouse-s42-bk_batch/z_mean.npy
9,camex_human_mouse,scvi,1,OK,None,nan,None,None,100.000000,training/runs/baselines_scvi/camex_human_mouse-s42-e100-nl10-bk_batch/z_mean.npy


## 8 — Latent-shape sanity sweep

Comparability gate: every saved `z_mean.npy` must have
`shape == (adata.n_obs, 10)` (the matrix's `n_latent` target),
contain no NaN/Inf, and have row order matching the source adata.

This cell catches the failure mode where a method silently ignored the
`n_latent=10` override — its `latent_shape[1]` would be its library
default. Such rows are flagged here as `OFF_TARGET`. The downstream
metrics step is responsible for handling non-comparable dims; the
runner's job is to be honest about it.

In [48]:
sanity_rows = []
for spec in DATASETS:
    for method in TIER1_METHODS:
        cp = expected_cache_path(spec, method, seed=42)
        if not cp.exists():
            continue
        z = np.load(cp, mmap_mode='r')
        finite = bool(np.isfinite(z).all())
        cfg = json.loads((cp.parent / 'config.json').read_text())
        target_dim = 10
        sanity_rows.append({
            'dataset':         spec.stem,
            'method':          method,
            'shape':           tuple(z.shape),
            'expected_n_obs':  cfg.get('n_obs'),
            'matches_n_obs':   z.shape[0] == cfg.get('n_obs'),
            'on_target_dim':   z.shape[1] == target_dim,
            'flag':            'OK' if (z.shape[1] == target_dim and finite) else 'OFF_TARGET',
            'finite':          finite,
            'first_3_rows_norm': float(np.linalg.norm(z[:3])),
            'last_3_rows_norm':  float(np.linalg.norm(z[-3:])),
        })
df_sanity = pd.DataFrame(sanity_rows)
df_sanity


,dataset,method,shape,expected_n_obs,matches_n_obs,on_target_dim,flag,finite,first_3_rows_norm,last_3_rows_norm
0,camex_human_monkey,pca,"(12608, 10)",12608.0,True,True,OK,True,61.922558,33.594334
1,camex_human_monkey,harmony,"(12608, 10)",12608.0,True,True,OK,True,46.787861,32.055820
2,camex_human_monkey,scvi,"(12608, 10)",NaN,False,True,OK,True,11.353698,4.755837
3,camex_human_monkey,scanvi,"(12608, 10)",12608.0,True,True,OK,True,11.880047,4.650715
4,camex_human_monkey,sysvi,"(12608, 10)",12608.0,True,True,OK,True,16.165216,10.670303
5,camex_human_mouse,pca,"(8091, 10)",8091.0,True,True,OK,True,62.175858,52.607105
6,camex_human_mouse,harmony,"(8091, 10)",8091.0,True,True,OK,True,60.539654,51.086292
7,camex_human_mouse,scvi,"(8091, 10)",NaN,False,True,OK,True,10.377608,8.678316
8,camex_human_mouse,scanvi,"(8091, 10)",8091.0,True,True,OK,True,9.934219,8.468666
9,camex_human_mouse,sysvi,"(8091, 10)",8091.0,True,True,OK,True,12.786907,17.051296


## 9 — What's next

- **Latents** are saved under `chimera/training/runs/baselines_<method>/...`.
  The structure is identical across methods so a downstream metrics step
  can `glob` them in one read.
- **`_summary.json`** at `chimera/training/runs/baselines_logs/_summary.json`
  is written by the orchestrator's CLI; the notebook's Section 7 above
  re-derives the same view from disk.
- **Per-method log files** at `chimera/training/runs/baselines_logs/<method>-<dataset>-s<seed>-bk_<bk>.log`
  capture full stdout/stderr from every subprocess (one log per
  invocation, not per epoch).
- **Deferred items**: CAMEX and SATURN remain Tier-2 stubs; see Section 6
  above for the gating steps.

The downstream metrics step lives outside this notebook — it loads the
cached latents, computes integration metrics (kNN-species-purity,
sil-species, sil-celltype, label-transfer accuracy, per-celltype kNN
mixing), and renders the cross-method comparison table.